## Pinecone
- 벡터 데이터 베이스
- 텍스트를 벡터로 바꿔 저장, 나중에 질문과 비슷한 데이터를 빠르게 찾아주는 DB

쿼리는 pinecone에게 검색해달라고 보낸 것? 질문이라고 보면 될듯.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
#!uv add pinecone

Resolved 122 packages in 980ms
   Building rag-two @ file:///D:/yujin0902/rag_one_2/rag_two
      Built rag-two @ file:///D:/yujin0902/rag_one_2/rag_two
 Downloaded pinecone
Prepared 6 packages in 891ms
Uninstalled 1 package in 9ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 6 packages in 240ms
 + h2==4.4.1
 + hpack==4.2.0
 + hyperframe==6.1.0
 + msgspec==0.21.1
 + pinecone==10.0.0
 ~ rag-two==0.1.0 (from file:///D:/yujin0902/rag_one_2/rag_two)


In [1]:
import os #파이썬의 os모듈을 가져오는 것
# pinecone에 연결 및 관리, pinecone 인덱스의 서버리스 설정지정
from pinecone import Pinecone, ServerlessSpec

In [4]:
import os

print(os.getenv("PINECONE_KEY") is not None)

True


In [ ]:
#  ***** Pinecone_인덱스생성.png 확인하기 *****

#pinecone에 연결
pc = Pinecone(api_key= os.getenv("PINECONE_KEY"))

index_name = "quickstart-index"

#index가 이미 있는지 확인
#인덱스가 없는 경우 새로 생성 (OpenAI text-embedding-3-small 기준 1536차원)
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name, #벡터 저장 공간 이름
        dimension=1536, #벡터의 숫자 개수
        metric="cosine", #백터 유사도 계산 방식
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

#인덱스 연결
index = pc.index(index_name)

In [6]:
# 예시 벡터 데이터 (id, vector, metadata)
# * 실제 서비스에서는 임베딩 모델(OpenAI 등)을 통해 생성된 벡터를 넣어야 함
dummy_vector = [0.01] * 1536  # 1536차원 가상 벡터

vectors = [
    {
        "id": "doc1",
        "values": dummy_vector,
        "metadata": {"text": "Pinecone은 벡터 데이터베이스입니다.", "category": "tech"}
    }
]

# 데이터 업로드 (Upsert)
index.upsert(vectors=vectors, namespace="example-namespace")

UpsertResponse(upserted_count=1)

In [8]:
#pinecone index upsert.png 이미지 확인

from openai import OpenAI
import time

openai_client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

#테스트할 문서, pinecone에 저장할 문서, 아직은 문자열 형태
documents = [
    {"id": "doc1", "text": "삼성 노트북은 2025년에 100만 대 판매되었습니다."},
    {"id": "doc2", "text": "LG 노트북은 2025년에 80만 대 판매되었습니다."},
    {"id": "doc3", "text": "Apple MacBook은 2025년에 120만 대 판매되었습니다."},
]

##문장을 벡터로 변환해서 pinecone에 저장
# openai의 embedding 모델 사용
response = openai_client.embeddings.create(
    model="text-embedding-3-small", #사용한 모델 
    input=[document["text"] for document in documents], #documents에서 text만 뽑아서 벡터로 변환
)

#pinecone에 넣을 형태로 데이터를 정리
vectors = [
    {
        "id": document["id"], #문서 식별자
        "values": embedding.embedding, #실제 임베딩 벡터
        "metadata": {"text": document["text"]}, #원본 텍스트 등의 추가 정보
    }
    #documents와 response.data 를 1대1로 묶어줌
    for document, embedding in zip(documents, response.data)
]

#pinecone에 저장
index.upsert(vectors=vectors)
time.sleep(2) #프로그램을 2초동안 잠시 멈추는

### 검색할 질문 만들기
question = "가장 많이 팔린 노트북은?"
#질문을 임베딩으로 변환
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

# pinecone에서 검색
result = index.query(
    vector=question_vector, #검색 기준이 되는 질문 벡터를 전달
    top_k=2, #가장 유사한 결과를 2개 가져오겠다.
    include_metadata=True, #검색 결과에 metadata도 같이 가져와라
)

#검색 결과 출력
print(f"질문: {question}\n") 
for match in result.matches:
    print(f"{match.id}: {match.metadata['text']} (유사도: {match.score:.4f})") 

질문: 가장 많이 팔린 노트북은?

doc1: 삼성 노트북은 2025년에 100만 대 판매되었습니다. (유사도: 0.5948)
doc2: LG 노트북은 2025년에 80만 대 판매되었습니다. (유사도: 0.5631)
